# 2.1 理论计算题
## 已知条件
输入特征：$C_{in} \times H_{in} \times W_{in}=3 \times 32 \times 32$
卷积核：共16个，单个卷积核尺寸 $3\times5\times5$
超参：$Padding(P)=2，Stride(S)=2$

## 1. 输出特征图尺寸计算
### 计算公式
$$
H_{out}=\left\lfloor \frac{H_{in}+2P-k_h}{S} \right\rfloor+1,\quad
W_{out}=\left\lfloor \frac{W_{in}+2P-k_w}{S} \right\rfloor+1
$$
输出通道数 = 卷积核数量。

### 代入数值
\[
\begin{align}
H_{out}&=\frac{32+2\times2-5}{2}+1=\frac{31}{2}+1=15+1=16 \\
W_{out}&=\frac{32+2\times2-5}{2}+1=16 \\
C_{out}&=16
\end{align}
\]
**输出尺寸：$\boldsymbol{16\times16\times16}$（通道×高×宽）**

## 2. 单个输出通道单个像素点乘次数
单个像素由完整卷积核与输入感受野做点积：
$$
\text{乘法次数}=C_{in}\times k_h\times k_w=3\times5\times5=75
$$
**需要75次乘法运算**

## 最终答案
1. 特征图尺寸：$\boldsymbol{16\times16\times16}$
2. 单点乘法次数：$\boldsymbol{75}$

# 2.2 编程题

In [3]:
import numpy as np

def max_pool2d_forward(input_data, kernel_size, stride=1, padding=0):
    """
    手动实现二维最大池化前向传播
    参数：
        input_data: 输入张量，形状为 [N, C, H, W]
                    N：batch大小，C：通道数，H：高度，W：宽度
        kernel_size: 池化核大小，int 或 (kh, kw)
        stride: 步幅，int 或 (sh, sw)
        padding: 填充大小，int 或 (ph, pw)
    返回：
        output: 池化输出，形状 [N, C, H_out, W_out]
    """
    # 1. 解析输入尺寸
    N, C, H_in, W_in = input_data.shape
    
    # 2. 统一参数格式（支持int输入）
    kh, kw = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
    sh, sw = (stride, stride) if isinstance(stride, int) else stride
    ph, pw = (padding, padding) if isinstance(padding, int) else padding

    # 3. 计算输出特征图尺寸
    H_out = (H_in + 2 * ph - kh) // sh + 1
    W_out = (W_in + 2 * pw - kw) // sw + 1

    # 4. 对输入进行零填充（上下左右填充）
    # pad 格式：((前,后), (上,下), (左,右)) → 对应 N, C, H, W
    input_padded = np.pad(input_data, 
                          pad_width=((0, 0), (0, 0), (ph, ph), (pw, pw)),
                          mode='constant')

    # 5. 初始化输出张量
    output = np.zeros((N, C, H_out, W_out), dtype=input_data.dtype)

    # 6. 遍历 batch、通道、输出高度、输出宽度 → 逐个窗口取最大值
    for n in range(N):          # 遍历每个样本
        for c in range(C):      # 遍历每个通道（池化通道独立）
            for h_out in range(H_out):
                for w_out in range(W_out):
                    # 计算当前窗口在填充后输入上的坐标
                    h_start = h_out * sh
                    h_end = h_start + kh
                    w_start = w_out * sw
                    w_end = w_start + kw

                    # 截取当前池化窗口
                    window = input_padded[n, c, h_start:h_end, w_start:w_end]
                    
                    # 最大池化：取窗口最大值
                    output[n, c, h_out, w_out] = np.max(window)

    return output


# ------------------- 测试代码 -------------------
if __name__ == "__main__":
    # 构造测试输入：1个样本，3个通道，8×8特征图
    test_input = np.random.randn(1, 3, 8, 8)
    
    # 池化参数
    kernel_size = 2
    stride = 2
    padding = 1

    # 手动实现的最大池化
    output = max_pool2d_forward(test_input, kernel_size, stride, padding)
    
    print("输入形状：", test_input.shape)
    print("输出形状：", output.shape)

输入形状： (1, 3, 8, 8)
输出形状： (1, 3, 5, 5)


# 3.1 理论计算题
## 已知条件
输入通道数、输出通道数均为 $C$，卷积**无偏置参数**。
卷积层参数量公式：
$$Param = C_{out} \times C_{in} \times k_h \times k_w$$

### 1. 单个 $5\times5$ 卷积参数量计算
$C_{in}=C,\ C_{out}=C,\ k_h=k_w=5$
$$
Param_{5\times5}=C \times C \times 5 \times 5 = \boldsymbol{25C^2}
$$

### 2. 两层串联 $3\times3$ 卷积总参数量计算
两层卷积输入输出通道均为 $C$：
- 第一层参数量：$P_1 = C\times C\times3\times3=9C^2$
- 第二层参数量：$P_2 = C\times C\times3\times3=9C^2$

总参数量：
$$
P_{total}=P_1+P_2=9C^2+9C^2=\boldsymbol{18C^2}
$$

## 最终答案
1. $5\times5$卷积参数量：$\boldsymbol{25C^2}$
2. 两层串联$3\times3$卷积总参数量：$\boldsymbol{18C^2}$

> 补充说明：两个3×3堆叠感受野等价单个5×5，参数量更小，多层激活提升非线性表达能力，为VGG设计思想。

# 3.2 编程题

In [4]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    """
    标准 NiN 块实现
    参数：
        in_channels: 输入通道数
        out_channels: 输出通道数
        kernel_size: 主卷积核大小
        stride: 主卷积步幅
        padding: 主卷积填充
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        self.block = nn.Sequential(
            # 第一层：普通卷积层 + ReLU
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(inplace=True),
            
            # 第二层：1×1 卷积 + ReLU
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            
            # 第三层：1×1 卷积 + ReLU
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


# ------------------- 测试代码 -------------------
if __name__ == "__main__":
    # 构造 NiN 块：输入3通道，输出16通道，卷积核5×5，步幅1，填充2
    nin_block = NiNBlock(in_channels=3, out_channels=16, kernel_size=5, stride=1, padding=2)
    
    # 构造测试输入：批量1，通道3，尺寸32×32
    test_input = torch.randn(1, 3, 32, 32)
    output = nin_block(test_input)
    
    print("输入形状:", test_input.shape)
    print("输出形状:", output.shape)

输入形状: torch.Size([1, 3, 32, 32])
输出形状: torch.Size([1, 16, 32, 32])


# 4.1 理论计算题
## 已知条件
样本：$x_1=2,\ x_2=4,\ x_3=6,\ x_4=8$
$\gamma=2,\ \beta=1,\ \epsilon=0$

### BN计算公式
$$
\mu=\frac1N\sum_{i=1}^N x_i,\quad
\sigma^2=\frac1N\sum_{i=1}^N(x_i-\mu)^2
$$
$$
\hat x_i=\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}},\quad
y_i=\gamma \hat x_i+\beta
$$

## 步骤1：求均值$\mu$
$$
\mu=\frac{2+4+6+8}{4}=\frac{20}{4}=5
$$

## 步骤2：求方差$\sigma^2$
$$
\begin{align}
\sigma^2&=\frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4}\\
&=\frac{9+1+1+9}{4}=\frac{20}{4}=5
\end{align}
$$

## 步骤3：标准化$\hat x_i=\dfrac{x_i-\mu}{\sqrt{\sigma^2}}$
$\sqrt{\sigma^2}=\sqrt5$
$$
\hat x_1=\frac{2-5}{\sqrt5}=-\frac{3}{\sqrt5},\quad
\hat x_2=\frac{4-5}{\sqrt5}=-\frac{1}{\sqrt5}
$$
$$
\hat x_3=\frac{6-5}{\sqrt5}=\frac{1}{\sqrt5},\quad
\hat x_4=\frac{8-5}{\sqrt5}=\frac{3}{\sqrt5}
$$

## 步骤4：仿射变换 $y_i=\gamma\hat x_i+\beta,\ \gamma=2,\beta=1$
$$
\begin{align}
y_1&=2\cdot\left(-\frac{3}{\sqrt5}\right)+1 = 1-\frac{6}{\sqrt5}\approx-1.6833\\
y_2&=2\cdot\left(-\frac{1}{\sqrt5}\right)+1 = 1-\frac{2}{\sqrt5}\approx0.1056\\
y_3&=2\cdot\left(\frac{1}{\sqrt5}\right)+1 = 1+\frac{2}{\sqrt5}\approx1.8944\\
y_4&=2\cdot\left(\frac{3}{\sqrt5}\right)+1 = 1+\frac{6}{\sqrt5}\approx4.6833
\end{align}
$$

## 最终结果
$$
\boldsymbol{y_1=1-\dfrac{6}{\sqrt5},\quad y_2=1-\dfrac{2}{\sqrt5},\quad y_3=1+\dfrac{2}{\sqrt5},\quad y_4=1+\dfrac{6}{\sqrt5}}
$$

# 4.2 编程题


In [5]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    """
    自定义残差块
    参数：
        in_channels: 输入通道数
        out_channels: 输出通道数
        stride: 主路径卷积步幅（默认1）
        use_1x1conv: 是否使用1×1卷积调整输入形状/通道
    """
    def __init__(self, in_channels, out_channels, stride=1, use_1x1conv=False):
        super(Residual, self).__init__()
        
        # 主路径：两个3×3卷积 + BN
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               padding=1, stride=stride, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, 
                               padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 捷径路径：1×1卷积（用于匹配通道/尺寸）
        self.conv3 = None
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, 
                                   kernel_size=1, stride=stride, bias=False)
        
        # 激活函数
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # 主路径前向传播
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        
        # 捷径路径
        if self.conv3 is not None:
            x = self.conv3(x)
        
        # 残差相加：f(x) + x
        out += x
        out = self.relu(out)
        
        return out


# ------------------- 测试代码 -------------------
if __name__ == "__main__":
    # 测试1：通道相同，无需1×1卷积
    block1 = Residual(16, 16)
    # 测试2：通道不同，需要1×1卷积
    block2 = Residual(16, 32, stride=1, use_1x1conv=True)
    
    x = torch.randn(1, 16, 32, 32)
    print("输入形状:", x.shape)
    print("输出形状1:", block1(x).shape)
    print("输出形状2:", block2(x).shape)

输入形状: torch.Size([1, 16, 32, 32])
输出形状1: torch.Size([1, 16, 32, 32])
输出形状2: torch.Size([1, 32, 32, 32])


# 5.1 理论计算题
## 1. 底层特征层小学习率/冻结、顶层输出层大学习率的原因
1. 预训练模型底层卷积网络在大数据集上学习到**边缘、纹理、轮廓等通用基础视觉特征**，该类特征对新任务依然适用，参数已经收敛优化。若设置大学习率会破坏已有有效权重，因此采用小学习率微调或直接冻结参数。
2. 顶层分类输出层是针对新数据集**随机初始化的全新参数**，没有经过预训练，需要较大学习率快速拟合新数据集的数据分布与分类标签，加快收敛。
3. 分层差异化学习率能够兼顾保留预训练学到的通用知识，同时适配下游新任务。

## 2. 目标数据集很小、和源数据集相似度高的防过拟合微调策略
1. **冻结整个主干特征提取网络**，仅训练末尾新增的分类输出层，不改动预训练权重，最大限度利用已有通用特征，避免小样本破坏主干参数引发过拟合。
2. 训练阶段使用**图像数据增广**（随机翻转、裁剪、色彩抖动等）扩充样本数量。
3. 引入正则化：在分类层添加Dropout、设置权重衰减L2正则，降低模型参数量冗余。
4. 采用小学习率训练+早停（Early Stopping），依据验证集损失提前终止训练，防止过拟合。

# 5.2 编程题

In [6]:
import torch
import torchvision.transforms as transforms

# 定义图像增广组合管道（严格满足4个要求）
train_aug = transforms.Compose([
    # 1. 随机裁剪+缩放：面积比例0.08~1.0，缩放至224×224
    transforms.RandomResizedCrop(size=224, scale=(0.08, 1.0)),
    
    # 2. 50%概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    
    # 3. 随机亮度、对比度、饱和度，变化范围0.5
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    
    # 4. 转换为PyTorch张量
    transforms.ToTensor()
])


# ------------------- 测试代码 -------------------
if __name__ == "__main__":
    from PIL import Image
    # 构造测试图像
    test_img = Image.new('RGB', (500, 500), color='red')
    
    # 应用增广
    aug_img = train_aug(test_img)
    
    print("增广后张量形状:", aug_img.shape)  # 输出: torch.Size([3, 224, 224])

增广后张量形状: torch.Size([3, 224, 224])


# 6.1 理论计算题
## 已知条件
真实框 $A=[x_{a1},y_{a1},x_{a2},y_{a2}]=[10,10,50,50]$
预测框 $B=[x_{b1},y_{b1},x_{b2},y_{b2}]=[30,30,70,70]$

IoU公式：
$$IoU=\frac{S_{inter}}{S_A+S_B-S_{inter}}$$

## 1. 分别计算A、B框面积
$$
\begin{align}
S_A &= (50-10)\times(50-10)=40\times40=1600 \\
S_B &= (70-30)\times(70-30)=40\times40=1600
\end{align}
$$

## 2. 求交集坐标
$$
\begin{align}
x_1 &= \max(x_{a1},x_{b1})=\max(10,30)=30 \\
y_1 &= \max(y_{a1},y_{b1})=\max(10,30)=30 \\
x_2 &= \min(x_{a2},x_{b2})=\min(50,70)=50 \\
y_2 &= \min(y_{a2},y_{b2})=\min(50,70)=50
\end{align}
$$
交集面积：
$$S_{inter}=(50-30)\times(50-30)=20\times20=400$$

## 3. 计算IoU
$$
IoU=\frac{400}{1600+1600-400}=\frac{400}{2800}=\frac{1}{7}\approx0.1429
$$

## 答案
$\boldsymbol{IoU=\dfrac17\approx0.1429}$

# 6.2 编程题


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, labels, epsilon=0.1, num_classes=None):
    """
    标签平滑后的交叉熵损失函数
    参数：
        logits: 模型输出 [batch_size, num_classes] （未经过 Softmax）
        labels: 真实标签 [batch_size] （类别索引，非 one-hot）
        epsilon: 平滑因子 ϵ，题目默认 0.1
        num_classes: 类别总数 K
    返回：
        标量损失值
    """
    if num_classes is None:
        num_classes = logits.size(1)  # 自动获取 K

    # 标签平滑公式：
    # 真实类概率 = 1 - ϵ
    # 其他类概率 = ϵ / (K-1)
    smooth_labels = torch.full_like(logits, epsilon / (num_classes - 1))
    smooth_labels.scatter_(1, labels.unsqueeze(1), 1.0 - epsilon)

    # 计算交叉熵（log_softmax + 点积）
    log_probs = F.log_softmax(logits, dim=1)
    loss = -(smooth_labels * log_probs).sum(dim=1).mean()

    return loss


# ------------------- 测试代码 -------------------
if __name__ == "__main__":
    # 模拟输出：batch=2，分类数 K=5
    logits = torch.randn(2, 5)
    labels = torch.tensor([0, 3])  # 真实标签

    loss = label_smoothing_cross_entropy(logits, labels, epsilon=0.1)
    print("标签平滑交叉熵损失 =", loss.item())

标签平滑交叉熵损失 = 2.788024425506592
